# 3.1 Build a Complete RAG Pipeline

**From documents to answers — the full Retrieval-Augmented Generation workflow.**

In this notebook you will:
- Understand how RAG combines retrieval with generation
- Learn what chunking is and why chunk size matters
- Build a complete RAG pipeline step by step
- Ask questions and get answers WITH source references
- Experiment with different chunk sizes to see their effect

> **Requirements:** You need a free Groq API key from [console.groq.com](https://console.groq.com)

## 1. Setup

In [ ]:
# Install all required packages:
# - langchain: framework for building LLM applications
# - langchain-groq: Groq LLM integration (fast, free tier)
# - langchain-community: community integrations (HuggingFace embeddings)
# - chromadb: vector database for storing embeddings
# - sentence-transformers: local embedding model
!pip install langchain langchain-groq langchain-community chromadb sentence-transformers -q

## 2. API Key Setup

We use **Groq** for the LLM — it's free and fast. Get your key at [console.groq.com](https://console.groq.com).

Embeddings run **locally** with sentence-transformers — no API key needed for those.

In [ ]:
import os
from # getpass removed import # getpass removed

# # getpass removed() hides your input so the key isn't visible in the notebook.
# The key is stored in an environment variable that the LLM client reads.
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = # getpass removed("Enter your Groq API key: ")

print("API key is set.")

## 3. Create Sample Documents

In a real project, you'd load documents from PDFs, databases, or APIs. For learning, we'll create travel policy documents as strings.

These documents represent the **knowledge base** that our RAG system will search through.

In [ ]:
# These are our "source documents" — the knowledge base for the RAG system.
# In production, these would come from PDFs, databases, or web scraping.

cancellation_policy = """
CANCELLATION AND REFUND POLICY

All tickets can be cancelled within 24 hours of booking for a full refund, regardless of fare class.
After 24 hours, cancellation fees apply based on fare class:
- Economy: $75 cancellation fee, refund to original payment method within 7-10 business days
- Premium Economy: $50 cancellation fee, refund within 5-7 business days
- Business Class: Free cancellation up to 48 hours before departure
- First Class: Free cancellation up to 24 hours before departure

Non-refundable tickets cannot be cancelled but can be changed for a $150 change fee plus fare difference.
Travel insurance provides full refund coverage for medical emergencies and family emergencies.
Group bookings (10+ passengers) have special cancellation terms — contact group desk for details.
"""

baggage_policy = """
BAGGAGE ALLOWANCE POLICY

Carry-on baggage: One personal item (purse, laptop bag) and one carry-on bag (max 55x40x20 cm, 7kg).
Checked baggage allowance varies by route and fare class:

Domestic flights:
- Economy: 1 bag, 23kg maximum
- Business: 2 bags, 32kg each

International flights:
- Economy: 2 bags, 23kg each
- Premium Economy: 2 bags, 28kg each
- Business: 3 bags, 32kg each
- First Class: 3 bags, 32kg each + 1 garment bag

Excess baggage fees: $50 per additional bag, $100 for overweight bags (23-32kg).
Sports equipment (skis, golf clubs, surfboards) counts as one checked bag if under 23kg.
Musical instruments can be carried in cabin if they fit in overhead bin, otherwise must be checked.
Fragile and valuable items should be carried in cabin — airline is not liable for damage to checked fragile items.
"""

loyalty_program = """
SKYREWARDS LOYALTY PROGRAM

Earning Miles:
- All flights: 1 mile per km flown (economy), 1.5x (premium economy), 2x (business), 3x (first class)
- Partner hotels: 500 miles per night
- Partner car rentals: 250 miles per rental
- Credit card spending: 1 mile per $1 spent on co-branded card

Tier Levels:
- Silver (25,000 miles/year): Priority check-in, 1 free checked bag
- Gold (50,000 miles/year): Lounge access, priority boarding, 2 free checked bags, free seat selection
- Platinum (100,000 miles/year): All Gold benefits + free upgrades when available, companion ticket annually

Redeeming Miles:
- Domestic flights: from 10,000 miles one-way
- International flights: from 25,000 miles one-way
- Upgrades: from 5,000 miles per segment
- Miles expire after 24 months of account inactivity
"""

booking_rules = """
BOOKING AND RESERVATION RULES

Online booking is available 330 days before departure.
Tickets must be purchased within 24 hours of reservation, otherwise the booking is automatically cancelled.
Seat selection is free for Business and First Class. Economy passengers can select seats for $15-45 depending on seat type.
Special meals (vegetarian, halal, kosher, gluten-free) must be requested at least 48 hours before departure.
Unaccompanied minors (ages 5-14) can travel with our escort service for $100 per flight segment.
Passengers with disabilities can request wheelchair assistance and priority boarding at no extra charge.
Name changes are not allowed after booking. Spelling corrections (up to 3 characters) can be made for free.
Group bookings (10+ passengers) receive a 10% discount and flexible payment terms.
"""

# Put all documents in a list for processing
raw_documents = [
    {"text": cancellation_policy, "source": "cancellation_policy.pdf"},
    {"text": baggage_policy, "source": "baggage_policy.pdf"},
    {"text": loyalty_program, "source": "loyalty_program.pdf"},
    {"text": booking_rules, "source": "booking_rules.pdf"},
]

print(f"Loaded {len(raw_documents)} documents")
for doc in raw_documents:
    print(f"  - {doc['source']}: {len(doc['text'])} characters")

## 4. What is Chunking?

Our documents are too long to embed as single pieces. We need to **chunk** them — split them into smaller, meaningful pieces.

Think of it like a **textbook**:
- The whole book is too big to search effectively
- Individual words are too small to be useful
- **Paragraphs** are the sweet spot — small enough to be specific, large enough to have context

**Chunk size** is critical:
- **Too small** (50 chars): loses context, e.g., "23kg each" without knowing what it refers to
- **Too large** (2000 chars): mixes topics, e.g., cancellation + baggage in one chunk
- **Just right** (200-500 chars): specific enough to match queries, enough context to be useful

**Chunk overlap** ensures we don't cut a sentence in half — each chunk shares some text with the previous one.

## 5. Chunk the Documents

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# RecursiveCharacterTextSplitter splits text at natural boundaries.
# It tries to split at "\n\n" first, then "\n", then " ", then individual characters.
# This keeps paragraphs and sentences intact as much as possible.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # Maximum characters per chunk
    chunk_overlap=50,    # Characters shared between adjacent chunks
    length_function=len  # Use character count (not token count)
)

# Convert our raw documents into LangChain Document objects and chunk them.
# Each chunk keeps track of which source document it came from via metadata.
all_chunks = []
for raw_doc in raw_documents:
    # Create a LangChain Document with metadata
    doc = Document(
        page_content=raw_doc["text"],
        metadata={"source": raw_doc["source"]}
    )
    # Split the document into chunks
    chunks = text_splitter.split_documents([doc])
    all_chunks.extend(chunks)

print(f"Split {len(raw_documents)} documents into {len(all_chunks)} chunks")
print(f"\nExample chunk (chunk #3):")
print(f"  Source: {all_chunks[3].metadata['source']}")
print(f"  Length: {len(all_chunks[3].page_content)} chars")
print(f"  Content: {all_chunks[3].page_content[:200]}...")

## 6. See How Chunk Size Affects Results

Let's compare small vs large chunks on the same document to understand the tradeoff.

In [ ]:
# Compare different chunk sizes on the same document
sample_text = cancellation_policy

for chunk_size in [100, 300, 600]:
    # Create a splitter with this chunk size
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=30
    )
    chunks = splitter.split_text(sample_text)
    
    print(f"\n{'='*60}")
    print(f"Chunk size: {chunk_size} chars → {len(chunks)} chunks")
    print(f"{'='*60}")
    for i, chunk in enumerate(chunks[:3]):  # Show first 3 chunks
        print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

print("\n---")
print("Small chunks (100): Very specific but may lose context")
print("Medium chunks (300): Good balance of specificity and context")
print("Large chunks (600): More context but may mix topics")

## 7. Embed and Store in ChromaDB

Now we embed all chunks and store them in a vector database for retrieval.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# HuggingFaceEmbeddings wraps sentence-transformers for use with LangChain.
# Runs locally — no API key needed, no cost.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a Chroma vector store from our chunks.
# Chroma automatically embeds each chunk using our embedding function.
# This is the "indexing" step — we do it once, then search many times.
vectorstore = Chroma.from_documents(
    documents=all_chunks,      # Our chunked documents
    embedding=embeddings,      # The embedding model to use
    collection_name="travel_rag"  # Name for the collection
)

print(f"Stored {len(all_chunks)} chunks in ChromaDB")
print("Ready for retrieval!")

## 8. Build the RAG Chain

Now we connect everything into a pipeline:

```
User Question → Retriever → Relevant Chunks → Prompt + LLM → Answer
```

Each component:
1. **Retriever**: searches the vector store for relevant chunks
2. **Prompt**: tells the LLM how to use the retrieved context
3. **LLM**: generates a natural language answer
4. **Parser**: extracts the text from the LLM response

In [ ]:
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Step 1: Create the Retriever ---
# The retriever searches our vector store and returns the top k most relevant chunks.
# k=4 means we retrieve 4 chunks to give the LLM enough context.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# --- Step 2: Create the Prompt ---
# This prompt tells the LLM exactly how to behave:
# - Use ONLY the provided context (no making things up)
# - Admit when the answer isn't in the context
prompt = ChatPromptTemplate.from_template("""
You are a helpful travel assistant. Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say "I don't have information about that in my documents."

Context:
{context}

Question: {question}

Answer:
""")

# --- Step 3: Create the LLM ---
# ChatGroq gives us access to Llama 3.3 70B — a powerful open-source model.
# temperature=0 makes responses deterministic (same input = same output).
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# --- Step 4: Create the Output Parser ---
# StrOutputParser extracts the text string from the LLM's response object.
parser = StrOutputParser()


def format_docs(docs):
    """
    Combine retrieved documents into a single string for the prompt.
    Each document is separated by double newlines for clarity.
    """
    return "\n\n".join(doc.page_content for doc in docs)


# --- Step 5: Chain Everything Together ---
# The | operator connects components in sequence:
# 1. retriever fetches relevant docs → format_docs joins them into a string
# 2. RunnablePassthrough() passes the question through unchanged
# 3. prompt fills in {context} and {question}
# 4. llm generates the answer
# 5. parser extracts the text
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

print("RAG chain is ready!")

## 9. Ask Questions and See Answers with Sources

Let's test the pipeline! We'll also show which source documents were used to generate each answer.

In [ ]:
# Let's ask a question and see the full pipeline in action
question = "What is the cancellation fee for an economy ticket?"

# First, let's see WHAT the retriever finds (the sources)
retrieved_docs = retriever.invoke(question)

print("STEP 1 — Retrieved Documents:")
print("=" * 60)
for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i+1} (from {doc.metadata['source']}):")
    print(f"  {doc.page_content[:150]}...")

# Now let's get the full answer from the chain
print("\n" + "=" * 60)
print("STEP 2 — Generated Answer:")
print("=" * 60)
answer = rag_chain.invoke(question)
print(f"\nQ: {question}")
print(f"A: {answer}")

# Show which sources contributed
print(f"\nSources: {set(doc.metadata['source'] for doc in retrieved_docs)}")

In [ ]:
# Test with multiple questions to see the pipeline handle different topics
questions = [
    "How many bags can I check on an international business class flight?",
    "How do I earn miles in the loyalty program?",
    "Can I change the name on my ticket?",
    "What is the meaning of life?",  # Not in our documents!
]

for question in questions:
    print(f"\nQ: {question}")
    
    # Get answer from the RAG chain
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    
    # Show which documents were retrieved
    docs = retriever.invoke(question)
    sources = set(doc.metadata["source"] for doc in docs)
    print(f"Sources: {sources}")
    print("-" * 50)

Notice how the last question ("meaning of life") gets a proper "I don't know" response — the LLM doesn't hallucinate because we told it to only use the provided context!

## 10. YOUR TURN: Experiment with Chunk Sizes

Try rebuilding the pipeline with different chunk sizes and see how the answers change.

Questions to consider:
- Does a smaller chunk size give more precise answers?
- Does a larger chunk size give more complete answers?
- What chunk size works best for your questions?

In [ ]:
# YOUR TURN: Change the chunk_size and chunk_overlap values,
# then run this cell and the next one to compare results.

# Try these combinations:
# - chunk_size=100, chunk_overlap=20  (very small chunks)
# - chunk_size=500, chunk_overlap=100 (larger chunks)
# - chunk_size=1000, chunk_overlap=200 (very large chunks)

experimental_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # ← CHANGE THIS
    chunk_overlap=100     # ← AND THIS
)

# Re-chunk the documents
experimental_chunks = []
for raw_doc in raw_documents:
    doc = Document(
        page_content=raw_doc["text"],
        metadata={"source": raw_doc["source"]}
    )
    chunks = experimental_splitter.split_documents([doc])
    experimental_chunks.extend(chunks)

# Rebuild the vector store with new chunks
experimental_vectorstore = Chroma.from_documents(
    documents=experimental_chunks,
    embedding=embeddings,
    collection_name="travel_rag_experiment"
)

# Rebuild the chain with the new retriever
experimental_retriever = experimental_vectorstore.as_retriever(search_kwargs={"k": 4})
experimental_chain = (
    {"context": experimental_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

print(f"New pipeline: {len(experimental_chunks)} chunks (chunk_size=500)")
print(f"Original pipeline: {len(all_chunks)} chunks (chunk_size=300)")

# Compare answers
test_q = "What are the baggage limits for international economy flights?"
print(f"\nQuestion: {test_q}")
print(f"\nOriginal (300 char chunks): {rag_chain.invoke(test_q)}")
print(f"\nExperimental (500 char chunks): {experimental_chain.invoke(test_q)}")

## Key Takeaways

1. **RAG = Retrieve + Generate**: find relevant documents first, then let the LLM answer using them
2. **Chunking** splits long documents into searchable pieces — chunk size is a critical parameter
3. **The prompt** is crucial — it tells the LLM to only use the context (preventing hallucination)
4. **Source tracking** lets users verify answers against original documents

**Next up:** In notebook **4.1**, we'll see where this naive RAG fails and learn **advanced techniques** (HyDE, re-ranking) to fix it!